# MedQuAD Medical Question-Answering Chatbot
## FLAN-T5 Model Development and Testing

**Model:** `google/flan-t5-small`  
**Task:** Medical Question Answering  
**Dataset:** MedQuAD  
**Model Architecture:** Sequence-to-Sequence Transformer  
**Framework:** Hugging Face Transformers

### Purpose

This notebook is used to develop, test, and evaluate the FLAN-T5 model
pipeline for the MedQuAD medical question-answering chatbot.

Reusable model logic is implemented inside the `models/` package,
while this notebook is used to execute experiments, inspect results,
and document the model-development process.

## 1. Experiment Overview

The initial experiment fine-tunes `google/flan-t5-small` using a
development sample of the MedQuAD dataset.

### Development Dataset

The processed development dataset contains 500 records:

| Dataset | Records | Percentage |
|---|---:|---:|
| Training | 400 | 80% |
| Validation | 50 | 10% |
| Test | 50 | 10% |
| **Total** | **500** | **100%** |

A group-aware splitting strategy was used so that identical questions
do not appear across different dataset splits.

The test dataset remains separate from model training and will be used
later for final evaluation.

## 2. Model Configuration

The initial model experiment uses FLAN-T5 Small.

### Configuration

| Parameter | Value |
|---|---|
| Model | `google/flan-t5-small` |
| Maximum input length | 64 tokens |
| Maximum target length | 512 tokens |
| Input feature | `input_text` |
| Target feature | `target_text` |

### Input Format

The model receives an instruction-formatted medical question:

`answer medical question: <question>`

### Target Format

The corresponding MedQuAD answer is used as the target sequence.

The 64-token input limit covers all questions observed in the
development sample. The 512-token target limit covers approximately
85.8% of sampled answers. Longer answers are truncated during
tokenization rather than removed from the dataset.

## 3. Environment Setup

The notebook is stored inside the `notebooks/` directory, while the
project's reusable Python modules are stored at the project root.

The project root is added to Python's module search path so modules
such as `models.train` can be imported from this notebook.

In [1]:
from pathlib import Path
import sys


# --------------------------------------------------------------
# Locate notebook and project directories
# --------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "notebooks"
    else NOTEBOOK_DIR
)


# --------------------------------------------------------------
# Add project root to Python import path
# --------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


# --------------------------------------------------------------
# Display environment information
# --------------------------------------------------------------

print(f"Notebook directory : {NOTEBOOK_DIR}")
print(f"Project root       : {PROJECT_ROOT}")

Notebook directory : d:\medquad_chatbot\medical_chatbot\notebooks
Project root       : d:\medquad_chatbot\medical_chatbot


## 4. Import Model Training Pipeline

Model-training functionality is maintained in `models/train.py`.

Keeping reusable model logic outside the notebook separates
implementation from experimentation and makes the same pipeline easier
to reuse later for AWS SageMaker training.

In [2]:
import importlib
import models.train

importlib.reload(models.train)

from models.train import (
    MODEL_NAME,
    MAX_INPUT_LENGTH,
    MAX_TARGET_LENGTH,
    load_training_data,
    load_tokenizer,
    load_model,
    tokenize_dataset
)

print("Model training module imported successfully.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Model training module imported successfully.


## 5. Verify Model Configuration

Verify that the notebook is using the expected FLAN-T5 model and
sequence-length configuration defined in `models/train.py`.

In [3]:
print(f"""
Model: {MODEL_NAME}\n
Maximum input length: {MAX_INPUT_LENGTH}\n
Maximum target length: {MAX_TARGET_LENGTH}
""")


Model: google/flan-t5-small

Maximum input length: 64

Maximum target length: 512



## 6. Load Training and Validation Data

The previously processed and feature-engineered training and
validation datasets are loaded from `data/processed/`.

Only the training and validation datasets are loaded at this stage.

- **Training data** is used to fine-tune FLAN-T5.
- **Validation data** is used to monitor model performance during development.
- **Test data** remains unseen until final model evaluation.

In [4]:
train_df, validation_df = (
    load_training_data()
)

print("Datasets loaded successfully.")

Datasets loaded successfully.


## 7. Dataset Verification

Before tokenization, verify the number of records, available features,
and basic integrity of the loaded datasets.

In [5]:
print("Dataset Sizes")
print("-" * 50)

print(
    f"Training records   : {len(train_df)}"
)

print(
    f"Validation records : {len(validation_df)}"
)


print("\nDataset Shapes")
print("-" * 50)

print(
    f"Training shape     : {train_df.shape}"
)

print(
    f"Validation shape   : {validation_df.shape}"
)


print("\nAvailable Columns")
print("-" * 50)

for column in train_df.columns:
    print(f"- {column}")

Dataset Sizes
--------------------------------------------------
Training records   : 400
Validation records : 50

Dataset Shapes
--------------------------------------------------
Training shape     : (400, 8)
Validation shape   : (50, 8)

Available Columns
--------------------------------------------------
- question
- answer
- source
- focus_area
- input_text
- target_text
- input_token_length
- target_token_length


## 8. Inspect a Training Example

FLAN-T5 is trained as a sequence-to-sequence model.

For each record:

**Input sequence**

`answer medical question: <question>`

**Target sequence**

The corresponding medical answer from MedQuAD.

Inspecting an example verifies that feature engineering produced the
expected model input and target format before tokenization.

In [6]:
sample = train_df.iloc[0]

print("ORIGINAL QUESTION")
print("-" * 70)
print(sample["question"])


print("\nMODEL INPUT")
print("-" * 70)
print(sample["input_text"])


print("\nTARGET ANSWER PREVIEW")
print("-" * 70)
print(
    sample["target_text"][:500]
)

ORIGINAL QUESTION
----------------------------------------------------------------------
What are the symptoms of Idiopathic inflammatory myopathy ?

MODEL INPUT
----------------------------------------------------------------------
answer medical question: What are the symptoms of Idiopathic inflammatory myopathy ?

TARGET ANSWER PREVIEW
----------------------------------------------------------------------
What are the signs and symptoms of Idiopathic inflammatory myopathy? The Human Phenotype Ontology provides the following list of signs and symptoms for Idiopathic inflammatory myopathy. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Autosomal dominant inheritanc


## 9. Model Data Validation

Before tokenization, perform basic checks to confirm that the training
and validation datasets contain the required model features and that
no records were lost during loading.

In [7]:
required_columns = [
    "input_text",
    "target_text"
]


for dataset_name, dataframe in [
    ("Training", train_df),
    ("Validation", validation_df)
]:

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    missing_values = (
        dataframe[required_columns]
        .isnull()
        .sum()
        .sum()
    )

    print(dataset_name)
    print("-" * 50)

    print(
        f"Records         : {len(dataframe)}"
    )

    print(
        f"Missing columns : {missing_columns}"
    )

    print(
        f"Missing values  : {missing_values}"
    )

Training
--------------------------------------------------
Records         : 400
Missing columns : []
Missing values  : 0
Validation
--------------------------------------------------
Records         : 50
Missing columns : []
Missing values  : 0


## 10. FLAN-T5 Tokenization

FLAN-T5 cannot process raw text directly. The input questions and
target answers must first be converted into token IDs using the
tokenizer associated with `google/flan-t5-small`.

### Tokenization Strategy

- Input maximum length: **64 tokens**
- Target maximum length: **512 tokens**
- Input truncation: **Enabled**
- Target truncation: **Enabled**
- Fixed-length padding: **Disabled**
- Dynamic batch padding: **Planned during training**

Dynamic padding will allow each training batch to be padded only to
the longest sequence within that batch rather than padding every
record to the global maximum sequence length.

### 10.1 Load FLAN-T5 Tokenizer

The tokenizer associated with `google/flan-t5-small` is loaded using
Hugging Face Transformers.

Using the tokenizer from the same pretrained model ensures that the
input and target sequences are encoded using the vocabulary expected
by FLAN-T5.

In [8]:
from models.train import load_tokenizer

tokenizer = load_tokenizer()

print("Tokenizer loaded successfully.")
print(f"Tokenizer: {tokenizer.__class__.__name__}")

Tokenizer loaded successfully.
Tokenizer: T5Tokenizer


### 10.2 Inspect Tokenization

Before tokenizing the complete training and validation datasets,
a single training example is tokenized to verify how FLAN-T5
converts the model input into numerical token IDs.

This check confirms that:

- The instruction-formatted question is correctly recognized.
- The tokenizer generates token IDs expected by FLAN-T5.
- The input sequence remains within the configured 64-token limit.

In [9]:
sample_text = train_df.loc[
    0,
    "input_text"
]

sample_tokens = tokenizer(
    sample_text,
    max_length=MAX_INPUT_LENGTH,
    truncation=True
)


print("Original Input")
print("-" * 60)
print(sample_text)


print("\nToken IDs")
print("-" * 60)
print(sample_tokens["input_ids"])


print("\nNumber of Tokens")
print("-" * 60)
print(len(sample_tokens["input_ids"]))


print("\nConfigured Maximum")
print("-" * 60)
print(MAX_INPUT_LENGTH)

Original Input
------------------------------------------------------------
answer medical question: What are the symptoms of Idiopathic inflammatory myopathy ?

Token IDs
------------------------------------------------------------
[1525, 1035, 822, 10, 363, 33, 8, 3976, 13, 27, 26, 23, 25216, 3, 15329, 82, 23599, 3, 58, 1]

Number of Tokens
------------------------------------------------------------
20

Configured Maximum
------------------------------------------------------------
64


### 10.3 Tokenize Training and Validation Datasets

The complete training and validation datasets are now converted into
FLAN-T5-compatible token representations.

For each example, tokenization produces:

- `input_ids` — numerical representation of the input question
- `attention_mask` — identifies tokens that should receive attention
- `labels` — numerical representation of the target medical answer

Input sequences are limited to 64 tokens and target sequences to
512 tokens. Sequences exceeding these limits are truncated.

Fixed-length padding is not applied at this stage because dynamic
padding will be used later during model training.

In [10]:
tokenized_train = tokenize_dataset(
    train_df,
    tokenizer
)

tokenized_validation = tokenize_dataset(
    validation_df,
    tokenizer
)

print("Tokenization completed successfully.")

Tokenization completed successfully.


### 10.4 Verify Tokenized Datasets

The tokenized datasets are checked to confirm that all training and
validation examples were preserved and that the required FLAN-T5
features were generated successfully.

In [11]:
print("Tokenized Features")
print("-" * 60)
print(list(tokenized_train.keys()))


print("\nDataset Sizes")
print("-" * 60)

print(
    f"Training examples   : "
    f"{len(tokenized_train['input_ids'])}"
)

print(
    f"Validation examples : "
    f"{len(tokenized_validation['input_ids'])}"
)

Tokenized Features
------------------------------------------------------------
['input_ids', 'attention_mask', 'labels']

Dataset Sizes
------------------------------------------------------------
Training examples   : 400
Validation examples : 50


### 10.5 Verify Sequence Lengths

The tokenized training data is checked to confirm that all input and
target sequences respect the configured maximum sequence lengths.

The model uses:

- Maximum input length: 64 tokens
- Maximum target length: 512 tokens

This verification ensures that truncation has been applied correctly
before the data is passed to FLAN-T5.

In [12]:
train_input_lengths = [
    len(tokens)
    for tokens in tokenized_train["input_ids"]
]

train_target_lengths = [
    len(tokens)
    for tokens in tokenized_train["labels"]
]


print("Training Sequence Lengths")
print("-" * 60)

print(
    f"Maximum input length     : "
    f"{max(train_input_lengths)}"
)

print(
    f"Maximum target length    : "
    f"{max(train_target_lengths)}"
)

print(
    f"Configured input limit   : "
    f"{MAX_INPUT_LENGTH}"
)

print(
    f"Configured target limit  : "
    f"{MAX_TARGET_LENGTH}"
)

Training Sequence Lengths
------------------------------------------------------------
Maximum input length     : 38
Maximum target length    : 512
Configured input limit   : 64
Configured target limit  : 512


### 10.6 Decode a Tokenized Example

A tokenized training example is decoded back into text to verify that
the numerical token representation corresponds to the expected
medical question and target answer.

This serves as a final sanity check before model initialization.

In [13]:
decoded_input = tokenizer.decode(
    tokenized_train["input_ids"][0],
    skip_special_tokens=True
)

decoded_target = tokenizer.decode(
    tokenized_train["labels"][0],
    skip_special_tokens=True
)


print("Decoded Input")
print("-" * 60)
print(decoded_input)


print("\nDecoded Target Preview")
print("-" * 60)
print(decoded_target[:500])

Decoded Input
------------------------------------------------------------
answer medical question: What are the symptoms of Idiopathic inflammatory myopathy ?

Decoded Target Preview
------------------------------------------------------------
What are the signs and symptoms of Idiopathic inflammatory myopathy? The Human Phenotype Ontology provides the following list of signs and symptoms for Idiopathic inflammatory myopathy. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Autosomal dominant inheritanc


## 11. FLAN-T5 Model Initialization

The pretrained `google/flan-t5-small` model will be used as the
foundation for fine-tuning on the MedQuAD medical question-answering
dataset.

FLAN-T5 is a sequence-to-sequence model. It receives the tokenized
medical question as input and learns to generate the corresponding
medical answer.

The pretrained model provides general language and instruction-following
capabilities, while fine-tuning adapts it to the medical question-answering
task.

### Model Configuration

- Model: `google/flan-t5-small`
- Architecture: Encoder-Decoder Transformer
- Task: Conditional Text Generation
- Input: Tokenized medical question
- Target: Tokenized medical answer
- Maximum input length: 64 tokens
- Maximum target length: 512 tokens

### 11.1 Import Model Initialization Function

The FLAN-T5 model-loading function is imported from the reusable
training module.

Model initialization requires a supported deep-learning backend.
Therefore, the execution environment will be verified before loading
the pretrained model.

In [14]:
from models.train import load_model

print("FLAN-T5 model loader imported successfully.")

FLAN-T5 model loader imported successfully.


### 11.2 Verify Deep-Learning Backend

Hugging Face tokenizers can operate without PyTorch, but loading and
fine-tuning the FLAN-T5 neural network requires a supported
deep-learning backend.

Before attempting model initialization, the current environment is
checked for PyTorch availability.

This allows the notebook to support both:

- Local preprocessing and tokenization
- Model initialization and training in a PyTorch-enabled environment

In [15]:
from transformers.utils import is_torch_available


torch_available = is_torch_available()


print("Model Environment")
print("-" * 60)

print(
    f"PyTorch available : "
    f"{torch_available}"
)

Model Environment
------------------------------------------------------------
PyTorch available : False


### 11.3 Initialize the Pretrained FLAN-T5 Model

The pretrained FLAN-T5 model is initialized only when PyTorch is
available in the execution environment.

If PyTorch is unavailable, model initialization is skipped without
interrupting the notebook. This allows preprocessing and tokenization
to remain executable locally while model training can later be
performed in a PyTorch-enabled environment such as AWS SageMaker.

In [16]:
model = None


if torch_available:

    model = load_model()

    print("FLAN-T5 model loaded successfully.")

    print(
        f"Model name  : "
        f"{MODEL_NAME}"
    )

    print(
        f"Model class : "
        f"{model.__class__.__name__}"
    )

else:

    print(
        "FLAN-T5 model initialization skipped."
    )

    print(
        "Reason: PyTorch is not available "
        "in the current environment."
    )

    print(
        "Model initialization and training "
        "will be performed in a "
        "PyTorch-enabled environment."
    )

FLAN-T5 model initialization skipped.
Reason: PyTorch is not available in the current environment.
Model initialization and training will be performed in a PyTorch-enabled environment.


### 11.4 Model Initialization Status

The FLAN-T5 tokenizer and tokenized MedQuAD datasets were successfully
prepared in the local development environment.

The local environment does not currently provide a usable PyTorch
backend. Therefore, the pretrained FLAN-T5 neural network is not
initialized locally.

Model initialization and fine-tuning will be executed in a
PyTorch-enabled training environment. This separation allows the
data-preparation pipeline to be validated locally before consuming
cloud compute resources for model training.

## 12. Training Configuration

The training configuration defines how the pretrained FLAN-T5 model
will be fine-tuned on the MedQuAD training dataset.

For the initial development experiment, a small training configuration
is used to validate the complete model-training pipeline before
scaling to the full MedQuAD dataset.

### Development Experiment

- **Model:** `google/flan-t5-small`
- **Training Records:** 400
- **Validation Records:** 50
- **Epochs:** 3
- **Training Batch Size:** 4
- **Validation Batch Size:** 4
- **Learning Rate:** 5e-5
- **Weight Decay:** 0.01
- **Maximum Input Length:** 64
- **Maximum Target Length:** 512

The test dataset remains excluded from training and validation and
will be used later for final model evaluation.

### 12.1 Define Training Hyperparameters

The initial hyperparameters are intentionally conservative because
the current 500-record dataset is a development sample used to test
the end-to-end training pipeline.

These values can later be adjusted based on validation performance
and experiments using the complete MedQuAD dataset.

In [17]:
training_config = {
    "num_train_epochs": 3,
    "learning_rate": 5e-5,
    "train_batch_size": 4,
    "eval_batch_size": 4,
    "weight_decay": 0.01,
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH
}


print("Training Configuration")
print("-" * 60)

for parameter, value in training_config.items():
    print(
        f"{parameter:<22}: {value}"
    )

Training Configuration
------------------------------------------------------------
num_train_epochs      : 3
learning_rate         : 5e-05
train_batch_size      : 4
eval_batch_size       : 4
weight_decay          : 0.01
max_input_length      : 64
max_target_length     : 512


### 12.2 Estimate Training Steps

The approximate number of optimization steps is calculated from the
training dataset size, batch size, and number of epochs.

This provides a simple estimate of the amount of training that will
occur during the development experiment.

In [18]:
import math


train_records = len(train_df)

batch_size = training_config[
    "train_batch_size"
]

num_epochs = training_config[
    "num_train_epochs"
]


steps_per_epoch = math.ceil(
    train_records / batch_size
)

total_training_steps = (
    steps_per_epoch * num_epochs
)


print("Training Step Estimate")
print("-" * 60)

print(
    f"Training records    : {train_records}"
)

print(
    f"Batch size          : {batch_size}"
)

print(
    f"Steps per epoch     : {steps_per_epoch}"
)

print(
    f"Number of epochs    : {num_epochs}"
)

print(
    f"Total training steps: {total_training_steps}"
)

Training Step Estimate
------------------------------------------------------------
Training records    : 400
Batch size          : 4
Steps per epoch     : 100
Number of epochs    : 3
Total training steps: 300


### 12.3 Define Model Artifact Directory

After fine-tuning, the trained model and tokenizer will be saved as
model artifacts.

Keeping model artifacts separate from source code and datasets makes
the project easier to manage and prepares the model for later
evaluation and deployment.

In [19]:
MODEL_OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "models"
    / "flan-t5-medquad"
)


print("Model Artifact Configuration")
print("-" * 60)

print(
    f"Output directory: "
    f"{MODEL_OUTPUT_DIR}"
)

Model Artifact Configuration
------------------------------------------------------------
Output directory: d:\medquad_chatbot\medical_chatbot\artifacts\models\flan-t5-medquad


### 12.4 Training Strategy

The initial FLAN-T5 fine-tuning experiment will use the following
strategy:

1. Load the pretrained `google/flan-t5-small` model.
2. Use the previously prepared tokenized training and validation data.
3. Apply dynamic padding during batch construction.
4. Fine-tune the model for three epochs.
5. Evaluate the model on the validation dataset during training.
6. Save the fine-tuned model and tokenizer as model artifacts.
7. Keep the test dataset completely separate until final evaluation.

The initial experiment is designed to validate the training pipeline
rather than maximize model performance.

After the pipeline is verified, training can be expanded to the
complete cleaned MedQuAD dataset and the hyperparameters can be tuned
using validation results.

## 13. FLAN-T5 Fine-Tuning Pipeline

This section prepares the tokenized MedQuAD datasets for FLAN-T5
fine-tuning.

All reusable training logic is maintained in `models/train.py`.
This notebook is used to execute the training pipeline, inspect
intermediate results, and document the experiment.

Before model training begins, the tokenized training and validation
datasets are validated to ensure that the model inputs and target
labels are correctly structured and aligned.

The held-out test dataset is not used during fine-tuning and will
be evaluated separately using `models/evaluate.py`.

### 13.1 Import the Updated Training Pipeline

The updated training module is reloaded to ensure that the notebook
uses the latest implementation from `models/train.py`.

The tokenized-dataset validation function will be used to verify the
training and validation datasets before fine-tuning.

In [20]:
import importlib
import models.train

importlib.reload(models.train)

from models.train import (
    MODEL_NAME,
    MAX_INPUT_LENGTH,
    MAX_TARGET_LENGTH,
    validate_tokenized_dataset
)

print("Updated training pipeline imported successfully.")

Updated training pipeline imported successfully.


### 13.2 Validate Tokenized Training and Validation Data

The tokenized datasets are validated before model training.

Each dataset must contain three aligned model features:

- `input_ids` — tokenized medical questions
- `attention_mask` — identifies valid input tokens
- `labels` — tokenized target medical answers

The number of examples in all three features must be identical.

In [21]:
train_validation_report = validate_tokenized_dataset(
    tokenized_train,
    "Training"
)

validation_validation_report = validate_tokenized_dataset(
    tokenized_validation,
    "Validation"
)

print("Tokenized datasets validated successfully.")

Tokenized datasets validated successfully.


### 13.3 Review Tokenized Dataset Validation Results

The validation reports summarize the structure of the tokenized
training and validation datasets before fine-tuning.

For each dataset, the report verifies:

- The type of tokenized data returned by the Hugging Face tokenizer
- The number of `input_ids`
- The number of `attention_mask` sequences
- The number of target `labels`
- Alignment between all model features

Successful alignment confirms that every tokenized medical question
has a corresponding attention mask and target answer.

In [23]:
print("Training Dataset")
print("-" * 60)

for key, value in train_validation_report.items():
    print(f"{key:<20}: {value}")


print("\nValidation Dataset")
print("-" * 60)

for key, value in validation_validation_report.items():
    print(f"{key:<20}: {value}")

Training Dataset
------------------------------------------------------------
dataset             : Training
data_type           : BatchEncoding
input_sequences     : 400
attention_masks     : 400
target_labels       : 400
aligned             : True

Validation Dataset
------------------------------------------------------------
dataset             : Validation
data_type           : BatchEncoding
input_sequences     : 50
attention_masks     : 50
target_labels       : 50
aligned             : True


### 13.4 Verify Fine-Tuning Data Readiness

Before proceeding to model fine-tuning, both the training and
validation datasets must pass the tokenized-data validation checks.

The datasets are considered ready for training when:

- Training features are correctly aligned
- Validation features are correctly aligned
- Each input sequence has a corresponding attention mask
- Each input sequence has a corresponding target label

If both datasets pass these checks, the tokenized data is considered
structurally ready for the FLAN-T5 fine-tuning pipeline.

In [24]:
training_ready = (
    train_validation_report["aligned"]
    and validation_validation_report["aligned"]
)

print("Fine-Tuning Data Readiness")
print("-" * 60)

print(
    f"Training data ready   : "
    f"{train_validation_report['aligned']}"
)

print(
    f"Validation data ready : "
    f"{validation_validation_report['aligned']}"
)

print(
    f"Ready for training    : "
    f"{training_ready}"
)

Fine-Tuning Data Readiness
------------------------------------------------------------
Training data ready   : True
Validation data ready : True
Ready for training    : True


## 14. Training Results

Training and validation results will be inspected to evaluate learning
behavior and identify potential overfitting or underfitting.

## 15. Test-Set Evaluation

After model development is complete, the held-out test dataset will
be used for final model evaluation.

Evaluation functionality will be implemented in `models/evaluate.py`.

## 16. Medical Question Prediction

The fine-tuned model will be tested on example medical questions to
inspect the quality, relevance, and safety of generated responses.

Inference functionality will be implemented in
`models/prediction.py`.

## 17. Experiment Summary

This section will summarize the final experiment, including:

- Model configuration
- Training behavior
- Validation performance
- Test-set performance
- Example generated responses
- Observed limitations
- Hallucination or factuality concerns
- Potential improvements